# Bonds and fixed income

**Prerequisites:** **05** (pricing fundamentals). Price **vanilla fixed**, a **floating (FRN-style) bond**, and an **amortizing** bond; request rate risk and spread-style metrics where the registry exposes them.


## Concept

- **Vanilla fixed:** PV from discount curve + schedule.
- **FRN:** floating leg needs a **forward curve** for projected index fixings.
- **Amortizing:** principal path feeds coupon bases.
- **Risk:** `dv01`, **modified duration**, **convexity**; **Z-spread** / **I-spread** appear when the pricer stack supports them for your spec.


In [1]:
import json
from datetime import date

from finstack.valuations import ValuationResult, price_instrument_with_metrics, validate_instrument_json
from finstack.core.market_data import DiscountCurve, ForwardCurve, MarketContext

print("Imports OK (bonds).")


Imports OK (bonds).


## Instrument JSON


In [2]:
AS_OF = date(2025, 1, 15)
AS_OF_STR = AS_OF.isoformat()

fixed_bond = {
    "type": "bond",
    "spec": {
        "id": "UST-FIXED-10Y",
        "notional": {"amount": "1000000", "currency": "USD"},
        "issue_date": "2024-01-15",
        "maturity": "2034-01-15",
        "discount_curve_id": "USD-OIS",
        "accrual_method": "Linear",
        "settlement_days": 1,
        "ex_coupon_days": 0,
        "cashflow_spec": {
            "Fixed": {
                "coupon_type": "Cash",
                "freq": {"count": 6, "unit": "months"},
                "dc": "Thirty360",
                "bdc": "following",
                "calendar_id": "weekends_only",
                "end_of_month": False,
                "payment_lag_days": 0,
                "rate": "0.0425",
                "stub": "None",
            }
        },
        "call_put": None,
        "credit_curve_id": None,
        "attributes": {"tags": [], "meta": {}},
        "pricing_overrides": {"quoted_clean_price": 98.75},
    },
}

frn_bond = {
    "type": "bond",
    "spec": {
        "id": "FRN-USD-SOFR",
        "notional": {"amount": "1000000", "currency": "USD"},
        "issue_date": "2024-01-15",
        "maturity": "2029-01-15",
        "discount_curve_id": "USD-OIS",
        "accrual_method": "Linear",
        "settlement_days": 2,
        "ex_coupon_days": 0,
        "cashflow_spec": {
            "Floating": {
                "coupon_type": "Cash",
                "freq": {"count": 3, "unit": "months"},
                "stub": "ShortFront",
                "rate_spec": {
                    "index_id": "USD-SOFR-3M",
                    "spread_bp": "150",
                    "gearing": "1",
                    "gearing_includes_spread": True,
                    "floor_bp": "0",
                    "all_in_floor_bp": None,
                    "cap_bp": None,
                    "index_cap_bp": None,
                    "dc": "Act360",
                    "bdc": "modified_following",
                    "calendar_id": "weekends_only",
                    "fixing_calendar_id": None,
                    "reset_freq": {"count": 3, "unit": "months"},
                    "reset_lag_days": 2,
                    "payment_lag_days": 0,
                    "end_of_month": False,
                },
            }
        },
        "call_put": None,
        "credit_curve_id": None,
        "attributes": {"tags": [], "meta": {}},
        "pricing_overrides": {},
    },
}

amort_bond = {
    "type": "bond",
    "spec": {
        "id": "AMORT-NOTE",
        "notional": {"amount": "1000000", "currency": "USD"},
        "issue_date": "2024-01-15",
        "maturity": "2029-01-15",
        "discount_curve_id": "USD-OIS",
        "accrual_method": "Linear",
        "settlement_days": 2,
        "ex_coupon_days": 0,
        "cashflow_spec": {
            "Amortizing": {
                "base": {
                    "Fixed": {
                        "coupon_type": "Cash",
                        "freq": {"count": 6, "unit": "months"},
                        "dc": "Thirty360",
                        "bdc": "following",
                        "calendar_id": "weekends_only",
                        "end_of_month": False,
                        "payment_lag_days": 0,
                        "rate": "0.045",
                        "stub": "ShortFront",
                    }
                },
                "schedule": {"LinearTo": {"final_notional": {"amount": "250000", "currency": "USD"}}},
            }
        },
        "call_put": None,
        "credit_curve_id": None,
        "attributes": {"tags": [], "meta": {}},
        "pricing_overrides": {},
    },
}

for label, payload in (("fixed", fixed_bond), ("frn", frn_bond), ("amort", amort_bond)):
    try:
        validate_instrument_json(json.dumps(payload))
        print(label, "JSON OK")
    except Exception as exc:
        print(label, "validate:", type(exc).__name__, exc)

bond_fixed_json = json.dumps(fixed_bond)
bond_frn_json = json.dumps(frn_bond)
bond_amort_json = json.dumps(amort_bond)


fixed JSON OK
frn JSON OK
amort JSON OK


## MarketContext


In [3]:
ois = DiscountCurve(
    "USD-OIS",
    AS_OF,
    [(0.0, 1.0), (0.5, 0.985), (1.0, 0.97), (3.0, 0.90), (5.0, 0.82), (10.0, 0.65)],
    day_count="act_365f",
)
sofr = ForwardCurve(
    "USD-SOFR-3M",
    0.25,
    [(0.0, 0.052), (1.0, 0.048), (3.0, 0.045), (5.0, 0.043), (10.0, 0.041)],
    base_date=AS_OF,
    day_count="act_360",
)
mc = MarketContext().insert(ois).insert(sofr)
market_json = mc.to_json()
print("curves in snapshot:", len(json.loads(market_json)["curves"]))


curves in snapshot: 2


## Pricing


In [4]:
for label, bjson in (
    ("fixed", bond_fixed_json),
    ("frn", bond_frn_json),
    ("amort", bond_amort_json),
):
    try:
        out = price_instrument_with_metrics(bjson, market_json, AS_OF_STR, model="discounting")
        print(label, ValuationResult.from_json(out))
    except Exception as exc:
        print(label, "price:", type(exc).__name__, exc)


fixed ValuationResult(id="UST-FIXED-10Y", price=987618.0556, currency=USD, metrics=0)
frn ValuationResult(id="FRN-USD-SOFR", price=1110810.7492, currency=USD, metrics=0)
amort ValuationResult(id="AMORT-NOTE", price=965324.2559, currency=USD, metrics=0)


## Metrics


In [5]:
metrics_req = ["dv01", "duration_mod", "convexity", "z_spread", "i_spread"]
for label, bjson in (("fixed", bond_fixed_json), ("frn", bond_frn_json)):
    try:
        out = price_instrument_with_metrics(
            bjson, market_json, AS_OF_STR, model="discounting", metrics=metrics_req
        )
        vr = ValuationResult.from_json(out)
        print("—", label)
        for m in metrics_req:
            v = vr.get_metric(m)
            if v is not None:
                print(f"  {m}: {v}")
    except Exception as exc:
        print(label, "metrics:", type(exc).__name__, exc)


— fixed
  dv01: 0.0
  duration_mod: 7.3947858236127315
  convexity: 64.60080131514711
  z_spread: 0.0015482582926960324
  i_spread: 0.001584513728382808
— frn
  dv01: 11.144958969205618
  duration_mod: 3.603871258501929
  convexity: 14.813370541938257
  z_spread: -0.004294443534439191
  i_spread: -0.0054922487564800435


## Price-from-metric overrides

`PricingOverrides.market_quotes` supports **nine mutually-exclusive price-driving fields** that short-circuit model pricing with a market quote. At most one may be set per instrument; `validate` rejects multiple.

| Field                      | Units                              | Driven through                  |
|----------------------------|------------------------------------|---------------------------------|
| `quoted_clean_price`       | % of par (e.g. `98.5`)             | accrued → dirty                 |
| `quoted_dirty_price_ccy`   | currency units                     | direct                          |
| `quoted_ytm`               | decimal (e.g. `0.055` = 5.5%)      | street-convention YTM inversion |
| `quoted_ytw`               | decimal                            | maturity-flow YTM (use OAS for callable exercise) |
| `quoted_z_spread`          | decimal (e.g. `0.0125` = 125bp)    | Z-spread over discount curve    |
| `quoted_oas`               | decimal                            | OAS over short-rate tree        |
| `quoted_discount_margin`   | decimal (FRNs)                     | additive margin on float leg    |
| `quoted_i_spread`          | decimal                            | YTM − par-swap-rate             |
| `quoted_asw_market`        | decimal                            | market-convention ASW inversion |

The legacy `quoted_spread_bp` field has been renamed to `cds_quote_bp` (CDS-only); the old JSON key is still accepted as a serde alias.

In [6]:
import copy


def price_with_override(overrides: dict) -> float:
    spec = copy.deepcopy(fixed_bond)
    spec["spec"]["pricing_overrides"] = overrides
    out = price_instrument_with_metrics(
        json.dumps(spec), market_json, AS_OF_STR, model="discounting"
    )
    return ValuationResult.from_json(out).price


# Baseline: no override, pure model PV.
baseline_bond = copy.deepcopy(fixed_bond)
baseline_bond["spec"]["pricing_overrides"] = {}
baseline = price_with_override({})
print(f"baseline model PV               : {baseline:,.2f} USD")

# 1) Override by quoted clean price (already used above with 98.75).
clean_pv = price_with_override({"quoted_clean_price": 99.5})
print(f"quoted_clean_price=99.5         : {clean_pv:,.2f} USD")

# 2) Override by quoted dirty price (currency units).
dirty_pv = price_with_override({"quoted_dirty_price_ccy": 985_000.00})
print(f"quoted_dirty_price_ccy=985000   : {dirty_pv:,.2f} USD")

# 3) Override by YTM (decimal).
ytm_pv = price_with_override({"quoted_ytm": 0.055})
print(f"quoted_ytm=0.055                : {ytm_pv:,.2f} USD")

# 4) Override by Z-spread (decimal, additive over the discount curve).
zspread_pv = price_with_override({"quoted_z_spread": 0.0075})
print(f"quoted_z_spread=0.0075 (75bp)   : {zspread_pv:,.2f} USD")

# 5) Override by I-spread.
ispread_pv = price_with_override({"quoted_i_spread": 0.0050})
print(f"quoted_i_spread=0.0050 (50bp)   : {ispread_pv:,.2f} USD")

# 6) Mutual-exclusivity check: setting two price drivers is rejected by validate.
try:
    validate_instrument_json(
        json.dumps(
            {
                **fixed_bond,
                "spec": {
                    **fixed_bond["spec"],
                    "pricing_overrides": {"quoted_ytm": 0.05, "quoted_z_spread": 0.01},
                },
            }
        )
    )
except Exception as exc:
    print(f"\nmutual-exclusivity check rejects dual drivers: {type(exc).__name__}")

baseline model PV               : 1,020,393.47 USD
quoted_clean_price=99.5         : 995,118.06 USD
quoted_dirty_price_ccy=985000   : 985,000.00 USD
quoted_ytm=0.055                : 912,402.83 USD
quoted_z_spread=0.0075 (75bp)   : 944,379.90 USD
quoted_i_spread=0.0050 (50bp)   : 963,042.15 USD


## Scenario price shocks

`scenario_price_shock_pct` applies a **multiplicative PV shock** on top of whatever pricing path was taken (model PV or quoted-price override). The engine applies the shock **exactly once** regardless of whether you call `value()` directly or go through `price_with_metrics`. This is the canonical lever for downside/upside revaluations in portfolio stress tests.

In [7]:
shock_pct = -0.10  # -10% price shock
shocked_pv = price_with_override(
    {"quoted_clean_price": 98.75, "scenario_price_shock_pct": shock_pct}
)
unshocked_pv = price_with_override({"quoted_clean_price": 98.75})
print(f"unshocked PV                    : {unshocked_pv:,.2f} USD")
print(f"shocked PV (-10%)               : {shocked_pv:,.2f} USD")
print(f"ratio (should be ~0.90)         : {shocked_pv / unshocked_pv:.6f}")

# Scenario shock composes with any price-from-metric override: here we pin the
# bond at a 75bp Z-spread and apply a +5% upside shock on top.
combo_pv = price_with_override(
    {"quoted_z_spread": 0.0075, "scenario_price_shock_pct": 0.05}
)
base_zspread_pv = price_with_override({"quoted_z_spread": 0.0075})
print(f"\nZ-spread PV                     : {base_zspread_pv:,.2f} USD")
print(f"Z-spread PV + 5% upside shock   : {combo_pv:,.2f} USD")
print(f"ratio (should be 1.05)          : {combo_pv / base_zspread_pv:.6f}")

unshocked PV                    : 987,618.06 USD
shocked PV (-10%)               : 888,856.25 USD
ratio (should be ~0.90)         : 0.900000

Z-spread PV                     : 944,379.90 USD
Z-spread PV + 5% upside shock   : 991,598.90 USD
ratio (should be 1.05)          : 1.050000


## Takeaways

- **One pipeline** prices fixed, float, and amortizing bonds once curves are wired (`USD-OIS` + `USD-SOFR-3M` for FRNs).
- **Metrics** are registry-driven: missing names are omitted rather than erroring.
- **Quoted clean** on the fixed bond pins a level so the engine can imply yields/spreads consistently with desk practice.
- **Nine price-driving overrides** (`quoted_clean_price`, `quoted_dirty_price_ccy`, `quoted_ytm/ytw`, `quoted_z_spread`, `quoted_oas`, `quoted_discount_margin`, `quoted_i_spread`, `quoted_asw_market`) short-circuit model pricing; validation rejects setting more than one at a time.
- **`scenario_price_shock_pct`** applies a multiplicative PV shock exactly once regardless of entry point, and composes cleanly with any quoted-price override.